In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install transformers timm -q

import torch
import torchvision
from transformers import DistilBertModel, DistilBertTokenizer

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [3]:
import torch
import torchvision.models as models
from transformers import DistilBertModel, DistilBertTokenizer

device = torch.device("cuda")

# ── 1. Image Encoder: ResNet50 (bỏ FC cuối) ──────────────────────
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
image_encoder = torch.nn.Sequential(*list(resnet.children())[:-1]).to(device)
image_encoder.eval()

# ── 2. Text Encoder: DistilBERT ───────────────────────────────────
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
text_encoder = DistilBertModel.from_pretrained("distilbert-base-uncased").to(device)
text_encoder.eval()

print(" Cả 2 model đã load lên GPU")

# ── 3. Test forward 1 batch giả ───────────────────────────────────
batch_size = 32

# Image batch: (32, 3, 224, 224)
dummy_images = torch.randn(batch_size, 3, 224, 224).to(device)

# Text batch
dummy_texts = ["This is a sample product title"] * batch_size
tokens = tokenizer(dummy_texts, return_tensors="pt",
                   padding=True, truncation=True, max_length=64).to(device)

with torch.no_grad():
    img_feats = image_encoder(dummy_images)
    img_feats = img_feats.flatten(1)                              # → (32, 2048)
    text_out  = text_encoder(**tokens)
    text_feats = text_out.last_hidden_state[:, 0, :]             # → (32, 768)

print(f" Image features: {img_feats.shape}")   # torch.Size([32, 2048])
print(f" Text features : {text_feats.shape}")  # torch.Size([32, 768])

# ── 4. Đo VRAM ────────────────────────────────────────────────────
allocated = torch.cuda.memory_allocated() / 1024**3
reserved  = torch.cuda.memory_reserved()  / 1024**3
print(f"\n VRAM đang dùng : {allocated:.2f} GB")
print(f" VRAM đã reserve: {reserved:.2f} GB")
print(f" VRAM T4 tổng   : ~15 GB")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 205MB/s]


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Cả 2 model đã load lên GPU
 Image features: torch.Size([32, 2048])
 Text features : torch.Size([32, 768])

 VRAM đang dùng : 0.37 GB
 VRAM đã reserve: 0.89 GB
 VRAM T4 tổng   : ~15 GB
